# 46 · Bộ mã hoá EHR của ta có bằng XGBoost không — **KHÔNG dùng ảnh**

## Vì sao đây là notebook gác cổng

`06b` lập kỷ luật này và **ta đã bỏ qua suốt Stage 4**: encoder của `H2` được xác nhận bằng shuffle
(`+0.0134`) **trước** mọi kết luận fusion, cốt để loại phản biện *"`H6` thất bại chỉ vì encoder của
nó kém"*.

Ta chưa từng chấm `EHRColTokBag` ở chế độ **EHR-only** một lần nào. Nên mọi câu đã viết —
`G_ehr = +0.0040`, *"họ `gate` không đọc EHR"*, *"vi khuẩn full mua đúng nhóm nhiễm trùng"* — đều
đứng trên một tiền đề **chưa kiểm**.

```
encoder cua ta ~ XGBoost  ->  moi so fusion tu 37/38*/44b/45 DOC DUOC
encoder KEM han           ->  ENCODER la nut that, va moi ket luan fusion bi CONFOUND
```

## XGBoost chạy **trong chính notebook này**

`nb30` đo `0.1199`, nhưng đó là notebook khác, tập đặc trưng khác, có thể split khác. Ta đã trả giá
ba lần cho việc so qua notebook (`mean_lin`, `mldec` v2-vs-v3, mốc `+0.101`).

Nên `C2` chạy XGBoost **ngay tại đây** trên **đúng** ma trận EHR mà `37`/`38*`/`44b` dùng, **đúng**
lát cắt, **đúng** hàm `ap_cols`. `0.1199` chỉ còn là mốc tham khảo.

## Bốn đầu đọc, từ đơn giản tới đúng-như-đang-dùng

```
xgb       XGBoost, 40 mo hinh nhi phan            <- moc
linear    Linear tren ma tran EHR (+ tui gop)     <- san duoi
mlp       MLP 1 lop an
tok       EHRColTokBag -> trung binh token -> Linear
mldec     40 query LOP cross-attend token EHR     <- DUNG BO MA HOA + DUNG CACH DOC cua 37/38*
```

`mldec` mới là con số quyết định: nó dùng **chính** `EHRColTokBag` và **chính** cách `MLDecoder` đọc
token, chỉ bỏ ảnh đi.

**Không dùng ảnh** ⇒ CPU hoặc GPU nhẹ, vài phút. Chạy song song mọi thứ khác.

## C0 · Config

In [ ]:
import os, json, math, time, re
import numpy as np, pandas as pd
import torch, torch.nn as nn

try:
    from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES46   = os.path.join(OUT_DIR, 'stage4_nb46')

def ensure_res():
    '''Goi TRUOC MOI lan ghi. Drive FUSE danh roi thu muc RONG — da dinh o 43/38c/38d/44b.'''
    os.makedirs(RES46, exist_ok=True); return RES46
ensure_res(); open(os.path.join(RES46, '_run_info.json'), 'w').write('{"notebook":"46"}')

SEED, SEEDS = 42, (42, 43, 44)
EPOCHS, LR, WD = 60, 3e-3, 1e-4        # asl40 @E=60 — cau hinh 35 da chot
D_ETOK = 256                           # y het 37/38*/44b
XGB_ROUNDS, XGB_DEPTH = 300, 6
MOC_NB30 = 0.1199                      # tham khao; C2 tu chay XGB de khoi so qua notebook
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'thiet bi: {dev} (khong dung anh — CPU cung chay duoc)')

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id']+LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15*len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL          # CUNG cach cat nhu 35/37/38*/43/44*
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
print(f'{len(ORDER):,} anh | khop {TF_.sum():,} | chon {SL.sum():,} | val {VA.sum():,}')
print(f'ghi vao {RES46}')

## C1 · EHR — **verbatim `37`**, cộng bản bỏ `micro_org_*` và nạp túi vi khuẩn của `38c`

Đúng ma trận mà `37`/`38*`/`44b` dùng. Không sửa gì để con số so được.

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()

def _mat(cols):
    m = (E_DF.set_index('study_id')[cols].reindex(_sid)
             .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    return np.nan_to_num(m, nan=0.0)

COLS = {'Q_CTX': [c for c in SPEC['Q_CTX'] if c in E_DF.columns],
        'Q_FULL': [c for c in SPEC['Q_FULL'] if c in E_DF.columns]}
EHR = {k: _mat(v) for k, v in COLS.items()}
# chuan hoa bang thong ke cua LAT KHOP — khong cham lat chon, khong cham val
EHR_MU, EHR_SD = {}, {}
EHR_CLIP = 10.0     # cot NHI PHAN cuc hiem lam chuan hoa no ra gia tri KHONG LO:
                    #   1 duong / 219,785  ->  sd 2.1e-3  ->  gia tri chuan hoa 468
                    # Probe tuyen tinh (35) chiu duoc — chi can hoc trong so nho. ATTENTION thi
                    # KHONG: `EHRColTok` dua thang x_j vao token (T_j = b_j + x_j*W_j), va score
                    # ti le voi M^2 nen bien do 30 cho score ~2e4 -> softmax bao hoa -> inf/NaN.
BIN_OF = {}
for k, m in EHR.items():
    # Cot NHI PHAN thi KHONG z-score. z-score bien gia tri "co" thanh sqrt((1-p)/p):
    #   p = 1/219,785 -> 468   |   p = 0.01 -> 9.9   |   p = 0.5 -> 1.0
    # Do chinh la nguon cua gia tri khong lo, va no la LOI BIEU DIEN chu khong phai
    # loi cua du lieu: cot chi bao khong co "do lech chuan" de chia.
    # `29b` one-hot top-K vi khuan (`micro_org_*`) + co comorbidity => rat nhieu cot cuc hiem.
    b_ = np.array([np.isin(np.unique(m[:, j]), (0.0, 1.0)).all() for j in range(m.shape[1])])
    BIN_OF[k] = b_
    mu = m[TF_].mean(0, keepdims=True); sd = m[TF_].std(0, keepdims=True) + 1e-6
    mu[:, b_] = 0.0; sd[:, b_] = 1.0                       # nhi phan: giu nguyen 0/1
    EHR_MU[k], EHR_SD[k] = mu, sd
    z = (m - mu) / sd
    _n_out = int((np.abs(z) > EHR_CLIP).sum())
    EHR[k] = np.clip(z, -EHR_CLIP, EHR_CLIP).astype('float32')  # kep chi con la chan cuoi
    print(f'  {k}: {int(b_.sum())}/{len(b_)} cot NHI PHAN (giu 0/1) | '
          f'|z| max {np.abs(z).max():6.1f} | kep {_n_out:,} o ({_n_out/z.size:.4%})')

# --- gom cot thanh 8 HO. Quy tac, khong phai tay; moi cot phai roi vao DUNG mot ho. ---
FAM_RE = [('ctx',    r'^ctx_'),
          ('demo',   r'^(age|gender)$'),   # nhan khau hoc: LUON co mat, khong phai phep do.
                                           # `29b` xep vao "gia tri" vi muc dich cua no la hoi
                                           # "loai benh nhan vs benh"; o day chung la HO rieng.
          ('mask',   r'_mask$'),
          ('n_lab',  r'^n_lab_'),
          ('dh_lab', r'^dh_lab_'),
          ('lab',    r'^lab_'),
          ('vital',  r'^vital_'),
          ('como',   r'^como_'),
          ('micro',  r'^micro_')]
def famsplit(cols):
    out, seen = {}, set()
    for nm, rx in FAM_RE:
        g = [i for i, c in enumerate(cols) if re.search(rx, c) and i not in seen]
        seen |= set(g)
        if g: out[nm] = np.array(g)
    du = [c for i, c in enumerate(cols) if i not in seen]
    assert not du, f'cot khong thuoc ho nao: {du[:10]}'
    return out
FAM = {k: famsplit(v) for k, v in COLS.items()}
for k in COLS:
    print(f'{k}: {len(COLS[k])} cot -> {len(FAM[k])} ho  ' +
          ' | '.join(f'{n}:{len(ix)}' for n, ix in FAM[k].items()))

# ================== bo cot one-hot + nap tui vi khuan (y het 38c C1) ==================
_MICRO_RE = re.compile(r'^micro_org_')
_COLS0  = {k: list(v) for k, v in COLS.items()}
_KEEP   = {k: [i for i, c in enumerate(v) if not _MICRO_RE.match(c)] for k, v in _COLS0.items()}
_n_drop = {k: len(_COLS0[k]) - len(_KEEP[k]) for k in _COLS0}
print('')
print(f'bo cot one-hot micro_org_*: {_n_drop}  (danh tinh gio di qua BAG)')
assert _n_drop['Q_FULL'] == 30, f"Q_FULL bo {_n_drop['Q_FULL']} cot, phai la 30"
COLS   = {k: [_COLS0[k][i] for i in _KEEP[k]] for k in _COLS0}
EHR    = {k: EHR[k][:, _KEEP[k]]    for k in _COLS0}
BIN_OF = {k: BIN_OF[k][_KEEP[k]]    for k in _COLS0}
FAM    = {k: famsplit(v) for k, v in COLS.items()}

P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
assert os.path.exists(P_MICRO), f'khong thay {P_MICRO} -> chay 29c truoc'
_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
V_MIC = int(json.load(open(P_VOCAB))['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(ORDER.study_id.to_numpy())
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b
X = EHR['Q_FULL']
print(f'X {X.shape} | tui V={V_MIC-1} dai toi da {MIC_L} | '
      f'anh co vi khuan {(MCNT>0).any(1).mean():.1%}')

# --- cho XGBoost: tui GOP thanh dem theo id (one-hot thua, nhung XGB an duoc) ---
XB = np.zeros((len(ORDER), V_MIC - 1), 'float32')
for r in range(len(ORDER)):
    m = MCNT[r] > 0
    if m.any(): np.add.at(XB[r], MIDS[r][m], MCNT[r][m])
XG = np.concatenate([X, XB], 1)
print(f'dac trung cho XGBoost: {XG.shape} (ma tran dac + tui gop thanh dem)')

## C2 · XGBoost — 40 mô hình nhị phân, **trong chính notebook này**

Có `resume` theo nhãn: đứt thì chạy lại, nó tiếp từ nhãn dở.

In [ ]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, sc = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-sc, kind='stable'); yy, ss = y[r], sc[r]
        tp, fp = np.cumsum(yy), np.cumsum(1-yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp/(tp+fp), tp/n1
        o[j] = float((np.diff(np.r_[0.0, rc])*pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

import xgboost as xgb
P_XGB = os.path.join(RES46, 'xgb_pred.npy')
PX = np.load(P_XGB) if os.path.exists(P_XGB) else np.full((VA.sum(), C), np.nan, 'float32')
_dtr = xgb.DMatrix(XG[TF_], label=None); _dva = xgb.DMatrix(XG[VA])
t0 = time.time()
for j in range(C):
    if not np.isnan(PX[:, j]).any(): continue
    npos = int(YM[TF_, j].sum())
    if npos < 5: PX[:, j] = float(npos) / max(TF_.sum(), 1); continue
    d = xgb.DMatrix(XG[TF_], label=YM[TF_, j])
    bst = xgb.train(dict(objective='binary:logistic', eval_metric='aucpr',
                         max_depth=XGB_DEPTH, eta=0.1, subsample=0.8, colsample_bytree=0.8,
                         tree_method='hist', device=dev, nthread=os.cpu_count() or 4),
                    d, num_boost_round=XGB_ROUNDS, verbose_eval=False)
    PX[:, j] = bst.predict(_dva)
    ensure_res(); np.save(P_XGB, PX)
    print(f'  [{j+1:>2}/{C}] {LAB40[j]:<28} n+={npos:>6,}  {(time.time()-t0)/60:5.1f} ph', flush=True)
AP_X = ap_cols(YM[VA], PX)
MAP_XGB = float(np.nanmean(AP_X))
print(f'\nXGBoost EHR-only: mAP anh {MAP_XGB:.4f} | ' +
      ' '.join(f'{t} {v:.4f}' for t, v in tiers(AP_X).items()))
print(f'  moc nb30 (notebook khac, dac trung khac): {MOC_NB30:.4f}')

## C3 · Bốn đầu đọc neural — `asl40`, **epoch cuối**, 3 seed

`mldec` là con số quyết định: **chính** `EHRColTokBag` và **chính** cách `MLDecoder` đọc token,
chỉ bỏ ảnh đi.

In [ ]:
class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44b.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        p = torch.sigmoid(x); pm = (p - s.clip).clamp(min=0)
        return -(y*torch.log(p.clamp(min=s.eps))*(1-p)**s.gp
                 + (1-y)*torch.log((1-pm).clamp(min=s.eps))*pm**s.gn).mean()

class MicroBag(nn.Module):
    '''Y HET 38c/38d/44b.'''
    def __init__(s, V, d):
        super().__init__()
        s.bag = nn.EmbeddingBag(V, d, mode='sum'); nn.init.normal_(s.bag.weight, std=d**-0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w): return s.mlp(s.bag(ids, per_sample_weights=w))

class ColTok(nn.Module):
    '''EHRColTokBag bo phan `const` — 217 cot + 1 tong hop + 1 token BAG.'''
    def __init__(s, d_in, d, V):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d**-0.5); s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.mb = MicroBag(V, d); s.bln = nn.LayerNorm(d)
    def forward(s, e, ids, w):
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
        return torch.cat([s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1)),
                          s.bln(s.mb(ids, w)).unsqueeze(1)], 1)

class Head(nn.Module):
    def __init__(s, kind, d_in, V, d=D_ETOK):
        super().__init__(); s.kind = kind
        if kind == 'linear': s.f = nn.Linear(d_in + V - 1, C)
        elif kind == 'mlp':  s.f = nn.Sequential(nn.Linear(d_in + V - 1, 512), nn.GELU(),
                                                 nn.Dropout(0.2), nn.Linear(512, C))
        else:
            s.tok = ColTok(d_in, d, V)
            if kind == 'tok': s.f = nn.Linear(d, C)
            else:                                   # `mldec`: 40 query LOP cross-attend token EHR
                s.q  = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
                s.at = nn.MultiheadAttention(d, 8, dropout=0.1, batch_first=True)
                s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
                s.ff = nn.Sequential(nn.Linear(d, 1024), nn.GELU(), nn.Linear(1024, d))
                s.w  = nn.Parameter(torch.randn(C, d) * d**-0.5); s.bo = nn.Parameter(torch.zeros(C))
    def forward(s, e, ids, w, xb):
        if s.kind in ('linear', 'mlp'): return s.f(torch.cat([e, xb], 1))
        t = s.tok(e, ids, w)
        if s.kind == 'tok': return s.f(t.mean(1))
        q = s.q.weight.unsqueeze(0).expand(e.shape[0], -1, -1)
        a, _ = s.at(s.n1(q), t, t, need_weights=False); q = q + a
        q = q + s.ff(s.n2(q))
        return (q * s.w).sum(-1) + s.bo

def fit(kind, seed):
    torch.manual_seed(seed)
    mu, sd = X[TF_].mean(0, keepdims=True), X[TF_].std(0, keepdims=True) + 1e-6
    xb_mu = XB[TF_].mean(0, keepdims=True); xb_sd = XB[TF_].std(0, keepdims=True) + 1e-6
    m = Head(kind, X.shape[1], V_MIC).to(dev)
    crit = ASL(gn=4, gp=0).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    # `tok`/`mldec` dung `ColTok` -> tensor [B, 217, 256]. O B=3434 rieng no da 770 MB, cong
    # cat/LayerNorm/ba phep chieu Q,K,V cua attention thi len vai GB -> bo cham nhu treo.
    # 37/38* chay dung lop nay o BATCH=120. `linear`/`mlp` khong dung ColTok nen an lo lon duoc.
    BS = {'linear': 4096, 'mlp': 4096, 'tok': 512, 'mldec': 512}[kind]
    idx = np.where(TF_)[0]; bs = int(min(BS, max(256, len(idx)//64)))
    n_step = math.ceil(len(idx)/bs)
    T = lambda a, dt=torch.float32: torch.as_tensor(a, dtype=dt, device=dev)
    t0 = time.time()
    for ep in range(EPOCHS):
        m.train(); perm = np.random.default_rng(seed*1000+ep).permutation(len(idx))
        for i in range(0, len(idx), bs):
            k = idx[perm[i:i+bs]]
            opt.zero_grad(set_to_none=True)
            crit(m(T((X[k]-mu)/sd), T(MIDS[k], torch.long), T(MCNT[k]),
                   T((XB[k]-xb_mu)/xb_sd)), T(YM[k])).backward()
            opt.step()
        sch.step()
        if ep == 0 or (ep+1) % 10 == 0:      # de biet no dang chay chu khong treo
            print(f'    {kind} s{seed} ep{ep+1:>2}/{EPOCHS}  bs={bs} '
                  f'{n_step} buoc/epoch  {(time.time()-t0)/60:.1f} ph', flush=True)
    m.eval(); out = {}
    with torch.no_grad():
        for nm, msk in (('va', VA), ('sl', SL)):
            ii = np.where(msk)[0]; ps = []
            for i in range(0, len(ii), bs):      # CUNG `bs` — 4096 o day cung bo bo nho
                k = ii[i:i+bs]
                ps.append(torch.sigmoid(m(T((X[k]-mu)/sd), T(MIDS[k], torch.long), T(MCNT[k]),
                                          T((XB[k]-xb_mu)/xb_sd))).cpu().numpy())
            out[nm] = np.concatenate(ps)
    return out['va'], out['sl']

P_RES = os.path.join(RES46, 'heads.csv')
R = pd.read_csv(P_RES) if os.path.exists(P_RES) else pd.DataFrame()
for kind in ('linear', 'mlp', 'tok', 'mldec'):
    for sd_ in SEEDS:
        if len(R) and ((R.dau_doc == kind) & (R.seed == sd_)).any(): continue
        t0 = time.time(); pv, ps = fit(kind, sd_)
        a = ap_cols(YM[VA], pv)
        R = pd.concat([R, pd.DataFrame([dict(dau_doc=kind, seed=sd_, mAP_anh=float(np.nanmean(a)),
            **tiers(a), sel=float(np.nanmean(ap_cols(YM[SL], ps))))])], ignore_index=True)
        ensure_res(); R.to_csv(P_RES, index=False)
        np.save(os.path.join(RES46, f'p_{kind}_s{sd_}.npy'), pv)
        print(f'{kind:<7} s{sd_}  mAP {R.mAP_anh.iloc[-1]:.4f}  ({time.time()-t0:.0f}s)')
print('\nxong')

## C4 · Phán quyết + ghi `REVIEW_EHR_ONLY.md`

In [ ]:
R = pd.read_csv(P_RES)
piv = R.pivot_table(index='dau_doc', values=['mAP_anh','head','medium','tail','sel'], aggfunc='mean')
piv['bien_do'] = R.groupby('dau_doc').mAP_anh.agg(lambda s: s.max()-s.min())
print('=== NEURAL (TB qua 3 seed) ===')
print(piv.reindex(['linear','mlp','tok','mldec']).to_string(float_format=lambda v: f'{v:.4f}'))
NOISE = float(piv.bien_do.max())
BEST  = float(piv.mAP_anh.max()); BEST_H = piv.mAP_anh.idxmax()
MLD   = float(piv.loc['mldec','mAP_anh'])
print(f'\nXGBoost (CUNG notebook, CUNG dac trung, CUNG split): {MAP_XGB:.4f}')
print(f'`mldec` (DUNG bo ma hoa cua 37/38*/44b)            : {MLD:.4f}')
print(f'tot nhat neural: {BEST_H} {BEST:.4f} | bien do seed {NOISE:.4f}')
d = MLD - MAP_XGB
print(f'\nmldec - XGB = {d:+.4f}')
print('  ---')
print(f'  |d| <= {max(NOISE, 0.005):.4f}  -> ENCODER DAT. Moi so fusion tu 37/38*/44b/45 DOC DUOC.')
print(f'  d << 0                 -> ENCODER la NUT THAT; moi ket luan fusion bi CONFOUND,')
print(f'                            va `G_ehr = +0.0040` la can duoi chu khong phai gia tri that.')
print(f'  KET LUAN: {"DAT" if abs(d) <= max(NOISE, 0.005) else ("ENCODER KEM HON" if d < 0 else "ENCODER HON XGB")}')

L = []; w = L.append
w('# REVIEW · bộ mã hoá EHR có bằng XGBoost không (`nb46`)'); w('')
w(f'`asl40 E={EPOCHS}` epoch cuối · seeds {list(SEEDS)} · val {int(VA.sum()):,} ảnh · '
  f'**không dùng ảnh**. XGBoost chạy **trong chính notebook này**, cùng đặc trưng/split/chỉ số.'); w('')
w('| đầu đọc | mAP ảnh | biên độ | tail |'); w('|---|---|---|---|')
w(f'| `xgb` | {MAP_XGB:.4f} | — | {tiers(AP_X)["tail"]:.4f} |')
for h in ('linear','mlp','tok','mldec'):
    r = R[R.dau_doc == h]
    w(f'| `{h}` | {r.mAP_anh.mean():.4f} | {r.mAP_anh.max()-r.mAP_anh.min():.4f} | {r["tail"].mean():.4f} |')
w(''); w('```'); w(f'mldec - XGB = {d:+.4f}   (bien do seed {NOISE:.4f})')
w(f'moc nb30 (notebook khac): {MOC_NB30:.4f}'); w('```'); w('')
w(f'**{"ĐẠT — mọi số fusion đọc được" if abs(d) <= max(NOISE,0.005) else "ENCODER LÀ NÚT THẮT — mọi kết luận fusion bị confound"}**'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **`mldec` ở đây bỏ ảnh đi**, nên nó không phải arm của `37`/`38*`. Nó chỉ trả lời *"bộ mã hoá '
  'có trích được thông tin nhãn từ EHR không"*, không nói gì về việc fusion dùng nó ra sao.')
w('- **XGBoost ăn tú gộp thành đếm**, còn neural ăn `EmbeddingBag`. Hai biểu diễn vi khuẩn khác '
  'nhau — công bằng theo nghĩa "mỗi bên dùng dạng hợp với nó", không phải cùng một đầu vào.')
w('- Một tập đặc trưng (`Q_FULL`), một loss, một split.')
TXT = '\n'.join(L)
ensure_res(); open(os.path.join(RES46, 'REVIEW_EHR_ONLY.md'), 'w', encoding='utf-8').write(TXT)
print('\n' + TXT)